In [1]:
from pathlib import Path

import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

from plant_shape_analysis import PlantSequencesDataset, LeafSequencesDataset
from plant_shape_analysis.vis.plot_functions import visualize_plant_sequence, visualize_leaf_sequence

%load_ext autoreload
%autoreload 2

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


### Plant sequences

In [ ]:
dataset_path = Path("../data/TrackPlant3D/versions")
plant_dataset = PlantSequencesDataset(
    dataset_path,
    use_ply=True,
    alignment_method="stem_based",
    estimate_normals=True,
    auto_download=False
)

Plot one plant example with normals

In [ ]:
sample_scan = plant_dataset[0]["timepoints"][0]
points = sample_scan["points"]
normals = sample_scan["normals"]

# Plot
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.normals = o3d.utility.Vector3dVector(normals)
# pcd.normalize_normals() # they're already normalized
scale = 0.5  # Adjust this number — smaller = shorter, larger = longer
pcd.normals = o3d.utility.Vector3dVector(
    [n * scale for n in pcd.normals]
)
o3d.visualization.draw_geometries(
        [pcd],
        point_show_normal=True,
        up=[0, 0, 1],
        front=[1, 0, 0],
        lookat=[0, 0, 0],
    )

Visualize a whole sequence with normals

In [ ]:
selected_sequences = [
"maize_control_plant1",
"maize_control_plant2",
"maize_control_plant3",
"sorghum_control_plant1",
"sorghum_highlight_plant2",
"sorghum_highlight_plant3",
"tobacco_control_plant1",
"tobacco_control_plant2",
"tobacco_shade_plant3",
"tomato1_control2_plant2",
"tomato1_heat_plant3",
"tomato2_control_plant3"
]

In [ ]:
for plant_ts in plant_dataset:
    if plant_ts["sequence_name"] in selected_sequences:
        visualize_plant_sequence(plant_ts, window_name=f"{plant_ts['sequence_name']}", spacing=100, normals_scale=0.2)
    break

### Compute normals from scratch and examine orientation

In [ ]:
dataset_path = Path("../data/TrackPlant3D/versions")
plant_dataset = PlantSequencesDataset(
    dataset_path,
    use_ply=True,
    alignment_method="stem_based",
    estimate_normals=False,
    auto_download=False
)

In [ ]:
sample_seq = plant_dataset.get_timeseries_by_sequence_name("maize_control_plant1")
sample_seq["timepoints"]

In [2]:
dataset_path = Path("../data/TrackPlant3D/versions")
leaf_dataset = LeafSequencesDataset(
    dataset_path,
    use_ply=True,
    apply_alignment=True,
    plant_alignment_method="stem_based",
    estimate_plant_normals=True,
    auto_download=False
)

Applying STEM_BASED alignment to 43 plant sequences...
STEM_BASED alignment complete. 43 sequences aligned.
Estimating normals for all sparse plants in the dataset...
Normal estimation complete.
Apply multi-stage PCA-based alignment to 218 leaf sequences...
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection detected: flipping first principal component
Reflection det

In [3]:
# TODO 1: fix visualization function needing transformations key-value pair of leafs
# TODO 2: fix transform axis in vis function
sample_seq = leaf_dataset.get_timeseries_by_sequence_name("maize_control_plant1_leaf1") 

In [ ]:
# for time in sample_seq["timepoints"]:
#     print(sample_seq["sequence_name"], sample_seq["leaf_id"])
#     print(time["day"])
#     output_folder = Path(f"../tmp/{sample_seq['sequence_name']}")
#     output_folder.mkdir(parents=True, exist_ok=True)
#     pcd = o3d.t.geometry.PointCloud(time["points"])

#     o3d.t.io.write_point_cloud(str(output_folder / f"day{time['day']}.ply"), pcd)

In [4]:
visualize_leaf_sequence(sample_seq, show_coordinate_frame=True, show_principal_axes=True)

#### Analyze individual alignment steps

In [ ]:
aligned_timepoints = leaf_dataset._prealign_with_leaf_tips(sample_seq['timepoints'])
aligned_seq = {
    "timepoints": aligned_timepoints,
    "sequence_name": sample_seq["sequence_name"],
    "leaf_id": sample_seq["leaf_id"]
}
visualize_leaf_sequence(aligned_seq, show_coordinate_frame=True)

In [ ]:
aligned_timepoints = leaf_dataset._align_leaves_to_xy_plane(sample_seq['timepoints'])
aligned_seq = {
    "timepoints": aligned_timepoints,
    "sequence_name": sample_seq["sequence_name"],
    "leaf_id": sample_seq["leaf_id"]
}
visualize_leaf_sequence(aligned_seq, show_coordinate_frame=True)

In [ ]:
# It seems this works but only on unaligned leaves
aligned_timepoints_consistent = leaf_dataset._enforce_temporal_normal_consistency(aligned_seq["timepoints"])
aligned_seq_consistent = {
    "timepoints": aligned_timepoints_consistent,
    "sequence_name": sample_seq["sequence_name"],
    "leaf_id": sample_seq["leaf_id"]
}
visualize_leaf_sequence(aligned_seq_consistent, show_coordinate_frame=True)

In [ ]:
def estimate_normals(sample_seq):
    for timepoint in sample_seq["timepoints"]:
        points = timepoint["points"]
        if points is not None and len(points) > 0:
            # Estimate normals using Open3D
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(points)
            pcd.estimate_normals()
            pcd.orient_normals_to_align_with_direction()
            pcd.orient_normals_consistent_tangent_plane(k=30)
            normals = np.asarray(pcd.normals)
            timepoint["normals"] = normals

# Estimate normals for each timepoint
#estimate_normals(sample_seq)

# Ensure temporal consistency of normals across timepoints
#leaf_dataset._enforce_temporal_normal_consistency(sample_seq["timepoints"])

# Visualize
visualize_leaf_sequence(sample_seq)

#### Check saved transformations and inverse

In [5]:
sample_seq_trans = leaf_dataset.transformations['maize_control_plant1_leaf1']
sample_seq_trans

[('prealign_tip',
  [{'stage': 'prealign_tip',
    'day': 0,
    'rotation': array([[1., 0., 0.],
           [0., 1., 0.],
           [0., 0., 1.]]),
    'translation': array([0., 0., 0.]),
    'center': array([ 0.10638537, -2.36826725,  4.31777356])},
   {'stage': 'prealign_tip',
    'day': 1,
    'rotation': array([[ 0.98882588,  0.        ,  0.14907506],
           [ 0.        ,  1.        ,  0.        ],
           [-0.14907506,  0.        ,  0.98882588]]),
    'translation': array([0., 0., 0.]),
    'center': array([ 3.05858848, -3.45927855, 10.35470494])},
   {'stage': 'prealign_tip',
    'day': 2,
    'rotation': array([[ 0.99933942,  0.        , -0.03634186],
           [ 0.        ,  1.        ,  0.        ],
           [ 0.03634186,  0.        ,  0.99933942]]),
    'translation': array([0., 0., 0.]),
    'center': array([ 5.31076198, -3.64464181, 15.3256996 ])},
   {'stage': 'prealign_tip',
    'day': 3,
    'rotation': array([[ 0.99639749,  0.        ,  0.08480588],
        

In [6]:
composed_matrices = LeafSequencesDataset.compose_transformations(sample_seq_trans)
composed_matrices

[{'day': 0,
  'composed_matrix': array([[ 0.93366231,  0.07465182,  0.35028815, -1.32861211],
         [ 0.07465182,  0.91599203, -0.39418994,  1.49512773],
         [-0.35028815,  0.39418994,  0.84965434, 18.22106674],
         [ 0.        ,  0.        ,  0.        ,  1.        ]]),
  'individual_stages': [{'stage': 'prealign_tip',
    'matrix': array([[1., 0., 0., 0.],
           [0., 1., 0., 0.],
           [0., 0., 1., 0.],
           [0., 0., 0., 1.]]),
    'rotation': array([[1., 0., 0.],
           [0., 1., 0.],
           [0., 0., 1.]]),
    'translation': array([0., 0., 0.]),
    'center': array([ 0.10638537, -2.36826725,  4.31777356])},
   {'stage': 'pca_align',
    'matrix': array([[1., 0., 0., 0.],
           [0., 1., 0., 0.],
           [0., 0., 1., 0.],
           [0., 0., 0., 1.]]),
    'rotation': array([[1., 0., 0.],
           [0., 1., 0.],
           [0., 0., 1.]]),
    'translation': array([0., 0., 0.]),
    'center': array([ 0.10638537, -2.36826725,  4.31777356])},

In [7]:
original_timepoints = LeafSequencesDataset.apply_inverse_transformations(
    sample_seq,
    sample_seq_trans
)
original_sequence = {
    "sequence_name": sample_seq["sequence_name"],
    "leaf_id": sample_seq["leaf_id"],
    "timepoints": original_timepoints,
}
original_sequence

{'sequence_name': 'maize_control_plant1_leaf1',
 'leaf_id': 1,
 'timepoints': [{'day': 0,
   'leaf_id': 1,
   'points': array([[ 4.59565009, -2.30458945,  9.38603553],
          [-6.39556553,  5.98172836, 19.29486219],
          [-7.27827785,  1.40173053,  7.63031377],
          ...,
          [ 2.54914528, -1.09324219, 12.42152542],
          [ 0.24252286, -3.06928341,  4.11992062],
          [ 0.84815705, -6.42107038, -4.77407333]], shape=(5594, 3)),
   'normals': array([[-0.51936611, -0.85448033, -0.01105445],
          [-0.85636321,  0.23215282,  0.46124519],
          [-0.52070261, -0.81908053,  0.2407818 ],
          ...,
          [-0.40776119, -0.89672293,  0.17210114],
          [-0.03200679, -0.95024965,  0.30984056],
          [-0.3324862 , -0.86645409,  0.37243823]], shape=(5594, 3)),
   'dense_points': array([[  9.50467367,  -8.84438018, -11.80062525],
          [  9.41883141,  -8.99613267, -11.89288221],
          [  9.41459115,  -9.09197261, -11.90217774],
          ...,

In [11]:
visualize_leaf_sequence(original_sequence, show_coordinate_frame=True)

In [ ]:
# Visualize unaligned leaf sequence
leaf_dataset2 = LeafSequencesDataset(
    dataset_path,
    use_ply=True,
    apply_alignment=False,
    plant_alignment_method="stem_based",
    estimate_plant_normals=True,
    auto_download=False
)
sample_seq2 = leaf_dataset2.get_timeseries_by_sequence_name("maize_control_plant1_leaf1") 
visualize_leaf_sequence(sample_seq2, show_coordinate_frame=True)

Applying STEM_BASED alignment to 43 plant sequences...
STEM_BASED alignment complete. 43 sequences aligned.
Estimating normals for all sparse plants in the dataset...
Normal estimation complete.


In [10]:
visualize_leaf_sequence(sample_seq2, show_coordinate_frame=True)